# Bayesian Weibull AFT survival analysis

This is an exploratory time-to-event analysis. `time` is follow-up in days; `DEATH_EVENT = 1` means an observed death and `DEATH_EVENT = 0` means right-censoring at the recorded follow-up. The classification notebooks exclude follow-up time; this survival model uses it explicitly.

## Censoring likelihood and model

For duration `t > 0`, Weibull shape `a > 0`, scale `lambda > 0`, and event indicator `d`, each observation contributes

`d * log f_W(t | a, lambda) + (1 - d) * log S_W(t | a, lambda)`.

The Weibull density is `f_W(t) = (a/lambda) (t/lambda)^(a-1) exp(-(t/lambda)^a)` and the survival function is `S_W(t) = exp(-(t/lambda)^a)`. Therefore, observed deaths contribute the event-time log density, while censored records contribute the log survival probability through their follow-up time.

The accelerated-failure-time model uses `log(lambda_i) = alpha + x_i beta` with standardized baseline predictors, `alpha ~ Normal(log(median follow-up), 1)`, `beta_j ~ Normal(0, 0.5)`, and `log(a) ~ Normal(log(1.2), 0.5)`. Priors are regularizing for this small cohort. `exp(beta_j)` is the multiplicative time ratio for a one-standard-deviation predictor increase.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from pro_gradu.classification import classification_features
from pro_gradu.data import load_dataset

SEED = 42
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'heart_failure_clinical_records_dataset.csv').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
df = load_dataset(PROJECT_ROOT / 'heart_failure_clinical_records_dataset.csv')
X, event = classification_features(df)
time = df['time'].to_numpy(dtype=float)
event = event.to_numpy(dtype='int8')
scaler = StandardScaler().fit(X)
x_std = scaler.transform(X)
n, p = x_std.shape
coords = {'feature': X.columns.to_list(), 'observation': np.arange(n)}
print(f'{n} records; {event.sum()} observed events; {n - event.sum()} right-censored records')

In [ ]:
with pm.Model(coords=coords) as survival_model:
    x_data = pm.Data('x', x_std, dims=('observation', 'feature'))
    log_shape = pm.Normal('log_shape', mu=np.log(1.2), sigma=0.5)
    shape = pm.Deterministic('shape', pm.math.exp(log_shape))
    intercept = pm.Normal('intercept', mu=np.log(np.median(time)), sigma=1.0)
    beta = pm.Normal('beta', mu=0, sigma=0.5, dims='feature')
    log_scale = intercept + x_data @ beta
    scale = pm.Deterministic('scale', pm.math.exp(log_scale), dims='observation')

    event_log_density = pm.logp(pm.Weibull.dist(alpha=shape, beta=scale), time)
    censor_log_survival = -((time / scale) ** shape)
    contribution = event * event_log_density + (1 - event) * censor_log_survival
    pm.Potential('right_censored_likelihood', contribution.sum())

    # Prior/predictive latent event times for covariates in this cohort.
    pm.Weibull('latent_event_time', alpha=shape, beta=scale, dims='observation')
    prior = pm.sample_prior_predictive(samples=500, random_seed=SEED)
    idata = pm.sample(
        draws=1500, tune=1500, chains=4, cores=1, target_accept=0.95,
        random_seed=SEED, progressbar=False, return_inferencedata=True,
    )
    posterior_predictive = pm.sample_posterior_predictive(
        idata, var_names=['latent_event_time'], random_seed=SEED,
        progressbar=False, return_inferencedata=True,
    )

## Prior predictive check

These are prior draws of latent event times for the observed covariate profiles. They are not censored follow-up times; censoring is handled in the likelihood above.

In [ ]:
prior_time = prior.prior_predictive['latent_event_time'].values.ravel()
pd.Series(prior_time, name='prior latent event time (days)').describe(percentiles=[.05, .5, .95, .99]).to_frame()

## Chain diagnostics and posterior summaries

Review R-hat and bulk/tail effective sample size for shape, intercept, and every predictor. The notebook flags whether R-hat is below 1.01 and effective sample sizes exceed 400. A flag is a diagnostic prompt, not proof the model is adequate.

In [ ]:
summary = az.summary(idata, var_names=['shape', 'intercept', 'beta'], hdi_prob=0.90)
summary

In [ ]:
diagnostics = az.summary(idata, var_names=['shape', 'intercept', 'beta'], kind='diagnostics')
checks = pd.Series({
    'all_rhat_below_1_01': bool((diagnostics.r_hat < 1.01).all()),
    'minimum_bulk_ess': float(diagnostics.ess_bulk.min()),
    'minimum_tail_ess': float(diagnostics.ess_tail.min()),
    'bulk_ess_above_400': bool((diagnostics.ess_bulk > 400).all()),
    'tail_ess_above_400': bool((diagnostics.ess_tail > 400).all()),
})
checks.to_frame('result')

## Posterior time ratios

A time ratio above one indicates a longer modeled event-time scale for a one-standard-deviation increase in the corresponding predictor, conditional on the other covariates. These are associations in this cohort, not causal effects.

In [ ]:
beta_draws = idata.posterior['beta'].stack(sample=('chain', 'draw')).transpose('sample', 'feature').values
time_ratio = np.exp(beta_draws)
ratio_summary = pd.DataFrame({
    'median': np.quantile(time_ratio, .5, axis=0),
    'lower_90': np.quantile(time_ratio, .05, axis=0),
    'upper_90': np.quantile(time_ratio, .95, axis=0),
}, index=X.columns)
ratio_summary.sort_values('median')

## Posterior survival curve and censor-aware empirical curve

The model curve averages posterior survival probabilities across the cohort covariate profiles. The empirical Kaplan-Meier curve accounts for right-censoring. A large discrepancy can indicate model misspecification; overlapping curves do not validate the model externally.

In [ ]:
def kaplan_meier(durations, observed):
    order = np.argsort(durations)
    durations = np.asarray(durations)[order]
    observed = np.asarray(observed)[order]
    unique_times = np.unique(durations)
    at_risk = len(durations)
    survival = 1.0
    out = []
    for t in unique_times:
        at_time = durations == t
        deaths = int(observed[at_time].sum())
        censored = int((~observed[at_time].astype(bool)).sum())
        if deaths:
            survival *= (1 - deaths / at_risk)
        out.append((t, survival))
        at_risk -= deaths + censored
    return pd.DataFrame(out, columns=['time', 'survival'])

post = idata.posterior.stack(sample=('chain', 'draw'))
shape_draws = post['shape'].transpose('sample').values[::5]
intercept_draws = post['intercept'].transpose('sample').values[::5]
beta_draws = post['beta'].stack(sample=('chain', 'draw')).transpose('sample', 'feature').values[::5]
scales = np.exp(intercept_draws[:, None] + beta_draws @ x_std.T)
time_grid = np.linspace(1, time.max(), 80)
survival_draws = np.stack([np.exp(-((t / scales) ** shape_draws[:, None])).mean(axis=1) for t in time_grid], axis=1)
model_curve = pd.DataFrame({
    'time': time_grid, 'median': np.quantile(survival_draws, .5, axis=0),
    'lower_90': np.quantile(survival_draws, .05, axis=0),
    'upper_90': np.quantile(survival_draws, .95, axis=0),
})
km_curve = kaplan_meier(time, event.astype(bool))
fig, ax = plt.subplots(figsize=(8, 5))
ax.fill_between(model_curve.time, model_curve.lower_90, model_curve.upper_90, alpha=.2, label='90% posterior interval')
ax.plot(model_curve.time, model_curve['median'], label='Weibull AFT posterior median')
ax.step(km_curve.time, km_curve.survival, where='post', label='Kaplan-Meier')
ax.set(xlabel='Follow-up / event time (days)', ylabel='Survival probability', ylim=(0, 1.02))
ax.legend()
fig.tight_layout()

## Posterior predictive check and limitations

Replicated latent event times can be summarized to inspect whether the fitted event-time scale is plausible under the Weibull assumption. They are latent death times, not observed follow-up times, because the censoring mechanism is not modeled. The Kaplan-Meier comparison provides a censor-aware descriptive check.

This analysis is exploratory. The Weibull distribution imposes a parametric hazard shape; results depend on that assumption and on regularizing priors. Censoring is treated as non-informative conditional on the included predictors. The small cohort limits precision, and this is not a validated clinical model.

In [ ]:
replicated = posterior_predictive.posterior_predictive['latent_event_time'].values.reshape(-1, n)
rep_summary = pd.DataFrame({
    'median': np.quantile(replicated, .50, axis=1),
    'p90': np.quantile(replicated, .90, axis=1),
    'p95': np.quantile(replicated, .95, axis=1),
})
rep_summary.describe(percentiles=[.05, .5, .95]).T